In [1]:
import pandas as pd
import numpy as np

In [3]:
df= pd.read_csv("movies.csv")
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
df.info

<bound method DataFrame.info of        movieId                               title  \
0            1                    Toy Story (1995)   
1            2                      Jumanji (1995)   
2            3             Grumpier Old Men (1995)   
3            4            Waiting to Exhale (1995)   
4            5  Father of the Bride Part II (1995)   
...        ...                                 ...   
62418   209157                           We (2018)   
62419   209159           Window of the Soul (2001)   
62420   209163                    Bad Poems (2018)   
62421   209169                 A Girl Thing (2001)   
62422   209171      Women of Devil's Island (1962)   

                                            genres  
0      Adventure|Animation|Children|Comedy|Fantasy  
1                       Adventure|Children|Fantasy  
2                                   Comedy|Romance  
3                             Comedy|Drama|Romance  
4                                           Comedy  
.

In [7]:
df.describe()

,movieId
count,62423.000000
mean,122220.387646
std,63264.744844
min,1.000000
25%,82146.500000
50%,138022.000000
75%,173222.000000
max,209171.000000


In [9]:
df.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [10]:
df.drop_duplicates(inplace=True)


In [11]:
df.drop_duplicates(subset="movieId", inplace=True)


In [12]:
df.isnull().sum()


movieId    0
title      0
genres     0
dtype: int64

In [13]:
df["genres"] = df["genres"].fillna("Unknown")


In [14]:
# Lowercase + remove spaces
df["genres"] = df["genres"].str.lower().str.strip()

# Convert to list
df["genres"] = df["genres"].str.split("|")


In [15]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),"[adventure, animation, children, comedy, fantasy]"
1,2,Jumanji (1995),"[adventure, children, fantasy]"
2,3,Grumpier Old Men (1995),"[comedy, romance]"
3,4,Waiting to Exhale (1995),"[comedy, drama, romance]"
4,5,Father of the Bride Part II (1995),[comedy]


In [16]:
df["genres"] = df["genres"].apply(lambda x: [g for g in x if g != "" and g != "unknown"])


In [17]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),"[adventure, animation, children, comedy, fantasy]"
1,2,Jumanji (1995),"[adventure, children, fantasy]"
2,3,Grumpier Old Men (1995),"[comedy, romance]"
3,4,Waiting to Exhale (1995),"[comedy, drama, romance]"
4,5,Father of the Bride Part II (1995),[comedy]


In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df["genres"])

genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_)
df_final = pd.concat([df, genre_df], axis=1)

print(df_final.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                              genres  (no genres listed)  \
0  [adventure, animation, children, comedy, fantasy]                   0   
1                     [adventure, children, fantasy]                   0   
2                                  [comedy, romance]                   0   
3                           [comedy, drama, romance]                   0   
4                                           [comedy]                   0   

   action  adventure  animation  children  comedy  crime  ...  film-noir  \
0       0          1          1         1       1      0  ...          0   
1       0          1          0         1       0      0  ...          0   
2       0     

In [19]:
df_final = df_final.drop(columns=["genres"])
df_final.reset_index(drop=True, inplace=True)


In [20]:
df_final.to_csv("clean_movies.csv", index=False)


In [21]:
df=pd.read_csv("clean_movies.csv")

In [22]:
df.head()

,movieId,title,(no genres listed),action,adventure,animation,children,comedy,crime,documentary,...,film-noir,horror,imax,musical,mystery,romance,sci-fi,thriller,war,western
0,1,Toy Story (1995),0,0,1,1,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,2,Jumanji (1995),0,0,1,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,3,Grumpier Old Men (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
3,4,Waiting to Exhale (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,1,0,0,0,0
4,5,Father of the Bride Part II (1995),0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

X = csr_matrix(df[genre_cols].values)

model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(X)

distances, indices = model.kneighbors(X[10], n_neighbors=10)

print(indices)

[[23130  9265    92  9534  9543 33589 33743  9583  9202 26186]]


In [26]:
def recommend_movies(movie_title, top_n=10):
    if movie_title not in df["title"].values:
        return "Movie not found!"
    
    idx = df[df["title"] == movie_title].index[0]
    distances, indices = model.kneighbors(X[idx], n_neighbors=top_n+1)
    
    movie_indices = indices[0][1:]
    
    return df[["title"]].iloc[movie_indices]

In [30]:
recommend_movies("Jumanji ")


'Movie not found!'